# 6.4 詞表要多大？


同一特徵寬度下，擴大詞表、加入更多合併項，可能讓句子用較少格，也需保存更多輸入特徵與輸出候選。詞表大小的取捨，是「這份原文變幾格」與「模型多存幾個數」一起看。

每token32特徵，V種輸入表有V×32格；不共享輸出則再加同樣大的表。先比較300、1000、3000種成本。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
width = 32
for vocab in [300, 1000, 3000]:
    embedding = vocab * width
    input_and_output = 2 * embedding
    print("詞表", vocab, "輸入參數", embedding, "輸入FP32 bytes", embedding * 4, "不共享輸入輸出參數", input_and_output)

詞表 300 輸入參數 9600 輸入FP32 bytes 38400 不共享輸入輸出參數 19200
詞表 1000 輸入參數 32000 輸入FP32 bytes 128000 不共享輸入輸出參數 64000
詞表 3000 輸入參數 96000 輸入FP32 bytes 384000 不共享輸入輸出參數 192000


輸入三個詞表預算，程序只計表格格數。V300時embedding9600、FP32數據38400bytes；V1000是32000與128000；V3000是96000與384000。不共享輸入輸出則分別19200、64000、192000格。FP32四bytes的約定見 [5.9](https://birdhackor.github.io/tiny-perceptron-vlm/5.9.html)，這些數字不含訓練歷史、梯度或其他層，所以不是整機記憶體預算。

成本另一邊是實際T。更多合併如果剛好覆蓋當前文本，token數可能減少，注意力T×T便更省。但字表大不保證這份驗證中文一定切得短：如果訓練tokenizer的資料主要是英文，新增片段可能幫不上中文。應拿同一份未參與字表訓練的原文，記錄實際token數、還原是否正確與模型品質，再看預算。

罕見大token可能很少受訓，小片段能跨詞共享例子卻用較長序列，沒有通用最佳V。已有byte/BPE比較，同一原文從3894格降2112格，模型參數從41824增57696。序列短和表格大同時發生，下一節再比較共同原文代價。

練習只把width從32改成64，先預測每項格數與bytes都翻倍，再執行核對；詞表V不變，所以你增加的是每token描述寬度。接著把V300改600，第一項也翻倍，但這次改變的是可用token種類，兩種成本變化相似，資料含義卻不同。

<details>
<summary>補充：實作約定與原始紀錄</summary>

用[6.3的那19篇驗證片段](https://birdhackor.github.io/tiny-perceptron-vlm/6.3.html)實際量一次，就能看到這個取捨。逐byte工具有256種原文byte，加八個結構項，共264項；BPE則共512項。同一批原文，前者需3,894個普通文字token，後者只需2,112個，平均每篇約205格降至111格；這裡尚未加開始與結束標記。但兩個同樣寬度32、一層的模型，參數卻由41,824個增加至57,696個。片段較短與模型較大同時發生，不能只挑其中一邊就說更省；下一節再把預測品質放進來。

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
